# Natural Language Processing with Disaster Tweets

Objective: Predict which Tweets are about real disasters and which ones are not

__This notebook contains the fitted baseline NLP model__

# NLP

Using NLP tutorial on Kaggle:
https://www.kaggle.com/code/philculliton/nlp-getting-started-tutorial/notebook

In [1]:
import pandas as pd
import numpy as np
from sklearn import feature_extraction, linear_model, model_selection, preprocessing

In [2]:
train = pd.read_csv('datasets/train.csv')
test = pd.read_csv('datasets/test.csv')
sample = pd.read_csv('datasets/sample_submission.csv')

In [3]:
train.shape

(7613, 5)

In [4]:
test.shape

(3263, 4)

In [5]:
train.isna().sum()

id             0
keyword       61
location    2533
text           0
target         0
dtype: int64

In [6]:
test.isna().sum()

id             0
keyword       26
location    1105
text           0
dtype: int64

In [7]:
train.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


## Disaster vs. NOT Disaster

In [8]:
example_disaster = train[train['target'] == 1]['text'].values[1]
example_disaster

'Forest fire near La Ronge Sask. Canada'

In [9]:
example_not_disaster = train[train['target'] == 0]['text'].values[7]
example_not_disaster

'Love skiing'

## Building vectors

The theory behind the model we'll build in this notebook is pretty simple: the words contained in each tweet are a good indicator of whether they're about a real disaster or not (this is not entirely correct, but it's a great place to start).

We'll use scikit-learn's __CountVectorizer__ to count the words in each tweet and turn them into data our machine learning model can process.

Note: a __vector__ is, in this context, a set of numbers that a machine learning model can work with. We'll look at one in just a second.


In [10]:
count_vectorizer = feature_extraction.text.CountVectorizer()
# Documentation https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html

In [11]:
# Let's work with the first 5 tweets of data
print(train['text'].values[0:5])

['Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all'
 'Forest fire near La Ronge Sask. Canada'
 "All residents asked to 'shelter in place' are being notified by officers. No other evacuation or shelter in place orders are expected"
 '13,000 people receive #wildfires evacuation orders in California '
 'Just got sent this photo from Ruby #Alaska as smoke from #wildfires pours into a school ']


In [12]:
## let's get counts for the first 5 tweets in the data
example_train_vectors = count_vectorizer.fit_transform(train["text"][0:5])
example_train_vectors.shape

(5, 54)

5 elements with 54 unique words distributed across them

In [13]:
print(len(count_vectorizer.get_feature_names()))
# count_vectorizer.get_feature_names()

54


C:\Users\macko\AppData\Roaming\Python\Python310\site-packages\sklearn\utils\deprecation.py:87: FutureWarning: Function get_feature_names is deprecated; get_feature_names is deprecated in 1.0 and will be removed in 1.2. Please use get_feature_names_out instead.
  warnings.warn(msg, category=FutureWarning)


Each unique word in the 5 tweets is assigned a number (alphabetical order)

In [14]:
count_vectorizer.vocabulary_

{'our': 34,
 'deeds': 12,
 'are': 5,
 'the': 49,
 'reason': 39,
 'of': 29,
 'this': 50,
 'earthquake': 13,
 'may': 25,
 'allah': 4,
 'forgive': 18,
 'us': 52,
 'all': 3,
 'forest': 17,
 'fire': 16,
 'near': 26,
 'la': 24,
 'ronge': 42,
 'sask': 44,
 'canada': 11,
 'residents': 41,
 'asked': 7,
 'to': 51,
 'shelter': 47,
 'in': 21,
 'place': 37,
 'being': 8,
 'notified': 28,
 'by': 9,
 'officers': 30,
 'no': 27,
 'other': 33,
 'evacuation': 14,
 'or': 31,
 'orders': 32,
 'expected': 15,
 '13': 1,
 '000': 0,
 'people': 35,
 'receive': 40,
 'wildfires': 53,
 'california': 10,
 'just': 23,
 'got': 20,
 'sent': 46,
 'photo': 36,
 'from': 19,
 'ruby': 43,
 'alaska': 2,
 'as': 6,
 'smoke': 48,
 'pours': 38,
 'into': 22,
 'school': 45}

Vector is a combination of the tweet number and a number assigned to the word in alphabetical order

In [15]:
print(example_train_vectors)

  (0, 34)	1
  (0, 12)	1
  (0, 5)	1
  (0, 49)	1
  (0, 39)	1
  (0, 29)	1
  (0, 50)	1
  (0, 13)	1
  (0, 25)	1
  (0, 4)	1
  (0, 18)	1
  (0, 52)	1
  (0, 3)	1
  (1, 17)	1
  (1, 16)	1
  (1, 26)	1
  (1, 24)	1
  (1, 42)	1
  (1, 44)	1
  (1, 11)	1
  (2, 5)	2
  (2, 3)	1
  (2, 41)	1
  (2, 7)	1
  (2, 51)	1
  :	:
  (2, 32)	1
  (2, 15)	1
  (3, 21)	1
  (3, 14)	1
  (3, 32)	1
  (3, 1)	1
  (3, 0)	1
  (3, 35)	1
  (3, 40)	1
  (3, 53)	1
  (3, 10)	1
  (4, 50)	1
  (4, 53)	1
  (4, 23)	1
  (4, 20)	1
  (4, 46)	1
  (4, 36)	1
  (4, 19)	2
  (4, 43)	1
  (4, 2)	1
  (4, 6)	1
  (4, 48)	1
  (4, 38)	1
  (4, 22)	1
  (4, 45)	1


In [16]:
example_train_vectors

<5x54 sparse matrix of type '<class 'numpy.int64'>'
	with 61 stored elements in Compressed Sparse Row format>

There are 61 stored elements in Compressed Sparse Row format, but only 54 unique words.

Below we will see what are the repeated words.

In [17]:
# Get the indices from the CSR matrix 
indices = example_train_vectors.indices
word_counts = {index: 0 for index in set(indices)} #initialize word counts

# Count occurrences of indices (representing words)
for index in indices:
    word_counts[index] += 1

# Filter out repeated words (counts greater than 1) and map back to actual words using the vocabulary
inv_vocabulary = {v: k for k, v in count_vectorizer.vocabulary_.items()}
repeated_words = {inv_vocabulary[index]: count for index, count in word_counts.items() if count > 1}

# Print the repeated words and their counts
for word, count in repeated_words.items():
    print(f"Word: {word}, Count: {count}")

Word: all, Count: 2
Word: are, Count: 2
Word: evacuation, Count: 2
Word: in, Count: 2
Word: orders, Count: 2
Word: this, Count: 2
Word: wildfires, Count: 2


In [18]:
## we use .todense() here because these vectors are "sparse" (only non-zero elements are kept to save space)
print(example_train_vectors[0].todense().shape)
print(example_train_vectors[0].todense())

(1, 54)
[[0 0 0 1 1 1 0 0 0 0 0 0 1 1 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 1 0
  0 0 0 1 0 0 0 0 0 0 0 0 0 1 1 0 1 0]]


Now, let's do this on the entire dataset - first __train__

In [19]:
## let's get counts for the first 5 tweets in the data
train_vectors = count_vectorizer.fit_transform(train["text"])
train_vectors.shape

(7613, 21637)

And __test__

In [20]:
test_vectors = count_vectorizer.transform(test["text"])

__We're using .transform() for test_vectors instead of .fit_transform()__

Using just .transform() makes sure that the tokens in the train vectors are the only ones mapped to the test vectors

i.e. that the train and test vectors use the same set of tokens.

# Fitting a model

As we mentioned above, we think the words contained in each tweet are a good indicator of whether they're about a real disaster or not. 

The presence of particular word (or set of words) in a tweet might link directly to whether or not that tweet is real.

What we're assuming here is a linear connection. So let's build a linear model and see!

## Ridge regression

Our vectors are really big, so we want to push our model's weights toward 0 without completely discounting different words
- ridge regression is a good way to do this.

In [21]:
clf = linear_model.RidgeClassifier()

Let's test our model and see how well it does on the training data. For this we'll use __cross-validation__, where we train on a portion of the known data, then validate it with the rest. 

If we do this several times (with different portions) we can get a good idea for how a particular model or method performs.

The metric for this competition is F1, so let's use that here.

In [23]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=6, shuffle=True, random_state=42)
scores = model_selection.cross_val_score(clf, train_vectors, train["target"], cv=kf, scoring="f1")
scores

array([0.74494706, 0.73247779, 0.73938224, 0.75171737, 0.71764706,
       0.73058485])

Results are pretty good. F1 scores for different splits span from 0.71 to 0.75 at best. 

## Fitting the model on train and predictions

In [24]:
clf.fit(train_vectors, train['target'])

RidgeClassifier()

In [25]:
sample.head()

,id,target
0,0,0
1,2,0
2,3,0
3,9,0
4,11,0


In [26]:
sample['target'] = clf.predict(test_vectors)

In [27]:
sample.head()

,id,target
0,0,0
1,2,1
2,3,1
3,9,0
4,11,1


In [28]:
sample.to_csv('submission.csv', index=False)